In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from src.util import *
from src.models.model import *
from src.config import *
from src.preprocess import *
from src.generate_operating_condition import *
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow import keras
from sklearn.model_selection import train_test_split

## 1. 데이터 구성 

#### 데이터로드

In [2]:
input_path="./dataset/processed/input/"
output_path="./dataset/processed/output/"
boundary_condition_path=input_path+"boundary/"
operating_condition_path=input_path+"operating/"

In [3]:
boundary_data=np.load(boundary_condition_path+'boundary_condition.npy')
operating_data=np.load(operating_condition_path+'operating_condition.npy')
label_data=np.load(output_path+'label.npy')

boundary_data.shape, operating_data.shape, label_data.shape

((600000,), (20, 4), (20, 600000))

In [4]:
simul_config = simulaion_config()
x_dim = simul_config.x_dim
y_dim = simul_config.y_dim
z_dim = simul_config.z_dim

#### 데이터 전처리

In [5]:
# 공간정보 데이터
boundary_data = np.array([boundary_data]*20)
boundary_data_temp = boundary_data.copy()
boundary_data = []
for i in boundary_data_temp:
    i = i.reshape(x_dim,y_dim,z_dim,1)
    boundary_data.append(i)
boundary_data = np.array(boundary_data).astype('float32')
boundary_data.shape

(20, 150, 100, 40, 1)

In [6]:
# 운전정보 데이터
char2idx,idx2char=generate_operating_condition()
operating_data=driving_division(operating_data,idx2char)
driving_config=driving_condition_max_value_config(operating_data)
operating_data=driving_scaling(operating_data,driving_config)

special_token=[]
special_value=[]
for i in range(len(operating_data)):
    special_token.append(operating_data[:,0][i])
    special_value.append(operating_data[:,1][i])
special_token=np.array(special_token)
special_value=np.array(special_value)
special_token.shape,special_value.shape

((20, 2), (20, 2))

In [7]:
# 레이블 데이터 스케일링
max_label = np.max(label_data)
min_label = np.min(label_data)

label_data = (label_data-min_label)/(max_label-min_label)
print(np.max(label_data), np.min(label_data))

# 레이블 데이터 구성
label_data_temp = label_data.copy()
label_data = []
for i in label_data_temp:
    i = i.reshape(x_dim,y_dim,z_dim,1)
    label_data.append(i)
label_data = np.array(label_data).astype('float32')
label_data.shape


1.0 0.0


(20, 150, 100, 40, 1)

#### 데이터 분할

In [8]:
train_ratio = 0.75
valid_ratio = 0.15
test_ratio = 0.10
first_split_size=1-train_ratio
second_split_size=test_ratio/(test_ratio+valid_ratio)

# 데이터분할
cnn_train, cnn_tv, label_train, label_tv = train_test_split(boundary_data, label_data, test_size=first_split_size, random_state=42)
cnn_val, cnn_test, label_val, label_test = train_test_split(cnn_tv, label_tv, test_size=second_split_size, random_state=42)
spe_tok_train, spe_tok_tv, val_tok_train, val_tok_tv = train_test_split(special_token, special_value, test_size=first_split_size, random_state=42)
spe_tok_val, spe_tok_test, val_tok_val, val_tok_test = train_test_split(spe_tok_tv, val_tok_tv, test_size=second_split_size, random_state=42)

# 시뮬레이션 정보 분할
simulation_info = pd.read_csv("src/simulation_info.csv")
simulation_info = simulation_info['inlet_velocity'].values
simulation_train,simulation_tv = train_test_split(simulation_info,test_size=first_split_size, random_state=42)
simulation_val,simulation_test = train_test_split(simulation_tv,test_size=second_split_size, random_state=42)

In [9]:
X_train = [cnn_train, spe_tok_train, val_tok_train]
X_test = [cnn_test, spe_tok_test, val_tok_test]
X_val = [cnn_val, spe_tok_val, val_tok_val]
Y_train = label_train
Y_test = label_test
Y_val = label_val
print("X_train ",X_train[0].shape)
print("X_val ",X_val[0].shape)
print("X_test ",X_test[0].shape)
print("Y_train ",Y_train.shape)
print("Y_val ",Y_val.shape)
print("Y_test ",Y_test.shape,'\n')

X_train  (15, 150, 100, 40, 1)
X_val  (3, 150, 100, 40, 1)
X_test  (2, 150, 100, 40, 1)
Y_train  (15, 150, 100, 40, 1)
Y_val  (3, 150, 100, 40, 1)
Y_test  (2, 150, 100, 40, 1) 



## 2. 모델 구성

#### 모델정의

In [10]:
model_config=get_transformer_config()

In [11]:
model_config

dimension_setting:
- 150
- 100
- 40
hidden_size: 256
maximum_position_encoding: 2000
mlp_dim: 1024
model_name: CFDUnet
num_heads: 8
num_layers: 12
rate: 0.2
x_dim: 150
y_dim: 100
z_dim: 40

In [12]:
model=CFDUnet(model_config)

#### 모델 컴파일

In [13]:
def L2_loss(y_true,y_pred):
    error = tf.math.square(y_true-y_pred)
    loss = tf.reduce_sum(error)
    return loss

In [14]:
initial_learning_rate = 1e-5

lr_schedule = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate, decay_steps=100000, decay_rate=0.96, staircase=True
)

model.compile(
    # loss=tf.keras.losses.MeanAbsoluteError(),
    loss=L2_loss,
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    metrics=["mae","mse"],
)

# model.compile(
#     loss=L2_loss,
#     optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
#     metrics=["mae","mse"],
# )

#### 모델 경로지정

In [15]:
# 모델 저장 경로정의
WEIGHT_PATH = 'checkpoints/'
model_name = 'transUnet/'
test_num = 4
TEST_DIR = 'TEST' + str(test_num) + '/'
# 경로지정
checkpoint_path = WEIGHT_PATH + model_name + TEST_DIR + "TEST" + str(test_num)
checkpoint_dir = os.path.dirname(checkpoint_path)

if os.path.exists(checkpoint_dir):
    print("{} -- Folder already exists \n".format(checkpoint_dir))
else:
    os.makedirs(checkpoint_dir, exist_ok=True)
    print("{} -- Folder create complete \n".format(checkpoint_dir))

checkpoints/transUnet/TEST4 -- Folder create complete 



#### callback 함수 지정

In [16]:
# EarlyStopping
earlystop_callback = EarlyStopping(monitor='val_loss', min_delta=0.00001, patience=200)

# Tensorboard
log_dir = "logs/fit/" + 'TEST' + str(test_num) + "" 

if os.path.exists(log_dir):
    print("{} -- Folder already exists \n".format(log_dir))
else:
    os.makedirs(log_dir, exist_ok=True)
    print("{} -- Folder create complete \n".format(log_dir))
    
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

# checkpoint
cp_callback = ModelCheckpoint(
    checkpoint_path, monitor='val_loss', verbose=1, save_best_only=True, save_weights_only=True)

logs/fit/TEST4 -- Folder create complete 



## 3. 학습

In [17]:
BATCH_SIZE = 1

In [18]:
history = model.fit(X_train, Y_train, 
                    validation_data=(X_val,Y_val),
                    batch_size=BATCH_SIZE, epochs=10000,
                    callbacks=[earlystop_callback, cp_callback, tensorboard_callback])

Epoch 1/10000
15/15 [==============================] - 20s 476ms/step - loss: 64492.7422 - mae: 0.2731 - mse: 0.1075 - val_loss: 29195.2598 - val_mae: 0.1703 - val_mse: 0.0487

Epoch 00001: val_loss improved from inf to 29195.25977, saving model to checkpoints/transUnet/TEST4\TEST4
Epoch 2/10000
15/15 [==============================] - 5s 339ms/step - loss: 26776.6289 - mae: 0.1716 - mse: 0.0446 - val_loss: 41362.7188 - val_mae: 0.2448 - val_mse: 0.0689

Epoch 00002: val_loss did not improve from 29195.25977
Epoch 3/10000
15/15 [==============================] - 5s 340ms/step - loss: 23560.6680 - mae: 0.1650 - mse: 0.0393 - val_loss: 38551.2383 - val_mae: 0.2401 - val_mse: 0.0643

Epoch 00003: val_loss did not improve from 29195.25977
Epoch 4/10000
15/15 [==============================] - 5s 339ms/step - loss: 22369.1660 - mae: 0.1611 - mse: 0.0373 - val_loss: 36506.0586 - val_mae: 0.2352 - val_mse: 0.0608

Epoch 00004: val_loss did not improve from 29195.25977
Epoch 5/10000
15/15 [===